# Caracterizacao inicial da qualidade do ar - Rio de Janeiro

## Objetivo desta etapa
Nesta etapa, definimos o escopo da analise para os dados diarios de qualidade do ar do Rio de Janeiro.

## Entregaveis
- Periodo de analise definido.
- Variaveis de poluentes selecionadas.
- Unidade temporal confirmada como diaria.
- Criterio de qualidade minima por variavel (cobertura).


## 1. Importacoes e configuracoes

Configuracao da fonte de dados, parametros de periodo e regra de cobertura minima.


In [1]:
from __future__ import annotations

from typing import List

import pandas as pd

# Fonte oficial da serie diaria consolidada do Rio de Janeiro.
URL_DADOS = (
    "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/"
    "refs/heads/Refactoring-And-Documentation/"
    "Data/IntermediaryData/MonitorAr/DailyQualiarRj/"
    "serie_diaria_qualidade_ar_rio_de_janeiro.parquet"
)

# Coluna temporal principal.
COLUNA_DATA = "data"

# Unidade temporal esperada na analise (D = diaria).
UNIDADE_TEMPORAL = "D"

# Cobertura minima exigida para uma variavel entrar nas proximas etapas.
COBERTURA_MINIMA = 0.80

# Lista guia para selecionar poluentes de interesse.
VARIAVEIS_POLUENTES_CANDIDATAS: List[str] = [
    "pm10",
    "pm2_5",
    "o3",
    "no2",
    "so2",
    "co",
    "pts",
    "no",
    "nox",
]

# Colunas normalmente nao consideradas como poluente em analise de exposicao.
VARIAVEIS_NAO_POLUENTES = {
    "chuva",
    "tem",
    "ur",
}


## 2. Carregamento e validacao inicial da base

Leitura da base consolidada e checagens basicas de estrutura.


In [2]:
# Le a base consolidada e interpreta a coluna de data como datetime.
df_rj = pd.read_parquet(URL_DADOS)

if COLUNA_DATA not in df_rj.columns:
    raise KeyError(f"A coluna obrigatoria '{COLUNA_DATA}' nao foi encontrada na base.")

# Padroniza data sem horario e ordena para facilitar validacoes temporais.
df_rj[COLUNA_DATA] = pd.to_datetime(df_rj[COLUNA_DATA], errors="coerce").dt.normalize()
df_rj = df_rj.dropna(subset=[COLUNA_DATA]).sort_values(COLUNA_DATA).reset_index(drop=True)

print(f"Dimensao carregada: {df_rj.shape[0]:,} linhas x {df_rj.shape[1]:,} colunas")
print(f"Periodo bruto da base: {df_rj[COLUNA_DATA].min():%Y-%m-%d} ate {df_rj[COLUNA_DATA].max():%Y-%m-%d}")
display(df_rj.head())


Dimensao carregada: 2,557 linhas x 12 colunas
Periodo bruto da base: 2012-01-01 ate 2018-12-31


,data,no,no2,so2,pm2_5,nox,ur,temp,o3,co,pm10,chuva
0,2012-01-01,5.762,22.376,2.667,14.508,28.102,92.521,25.556,22.905,0.397,22.838,228.2
1,2012-01-02,25.347,29.317,2.290,8.158,54.670,95.074,22.264,13.924,0.311,14.965,371.0
2,2012-01-03,23.367,28.593,3.704,9.875,51.883,73.642,25.091,15.997,0.243,26.374,0.2
3,2012-01-04,28.645,36.209,3.177,14.823,64.833,72.631,26.083,23.245,0.267,35.924,0.4
4,2012-01-05,17.859,31.785,3.113,11.958,49.605,74.860,26.661,36.343,0.250,31.761,0.0


## 3. Definicao do escopo temporal

Fixa o periodo de analise e garante unidade temporal diaria.


In [3]:
# Remove duplicatas de data, se existirem, mantendo a primeira ocorrencia.
# Como a serie diaria da cidade deve ter uma linha por dia, duplicatas sao inconsistencias.
duplicatas_data = df_rj.duplicated(subset=[COLUNA_DATA], keep=False).sum()
if duplicatas_data > 0:
    print(f"Aviso: foram encontradas {duplicatas_data} linhas com data duplicada. Mantendo a primeira por data.")
    df_rj = df_rj.drop_duplicates(subset=[COLUNA_DATA], keep="first").copy()

# Confere continuidade do calendario diario dentro do periodo definido.
calendario_esperado = pd.date_range(start=df_rj[COLUNA_DATA].min(), end=df_rj[COLUNA_DATA].max(), freq=UNIDADE_TEMPORAL)
datas_observadas = pd.DatetimeIndex(df_rj[COLUNA_DATA].dropna().unique()).sort_values()
datas_faltantes = calendario_esperado.difference(datas_observadas)

print(f"Periodo de analise: {df_rj[COLUNA_DATA].min()} ate {df_rj[COLUNA_DATA].max()}")
print(f"Unidade temporal definida: diaria ({UNIDADE_TEMPORAL})")
print(f"Total de dias esperados: {len(calendario_esperado):,}")
print(f"Dias observados na base: {len(datas_observadas):,}")
print(f"Dias faltantes no calendario: {len(datas_faltantes):,}")

if len(datas_faltantes) > 0:
    print("Primeiros dias faltantes:", [d.strftime("%Y-%m-%d") for d in datas_faltantes[:10]])


Periodo de analise: 2012-01-01 00:00:00 ate 2018-12-31 00:00:00
Unidade temporal definida: diaria (D)
Total de dias esperados: 2,557
Dias observados na base: 2,557
Dias faltantes no calendario: 0


## 4. Definicao das variaveis de poluentes

Seleciona as variaveis de poluentes que estao presentes e disponiveis para analise.


In [4]:
# Busca colunas numericas, pois apenas elas entram em correlacao e analise quantitativa.
colunas_numericas = df_rj.select_dtypes(include="number").columns.tolist()

# Prioriza uma lista guia de poluentes conhecida.
variaveis_poluentes = [
    coluna for coluna in VARIAVEIS_POLUENTES_CANDIDATAS if coluna in colunas_numericas
]

# Fallback: se nenhum candidato estiver presente, usa numericas excluindo nao-poluentes conhecidas.
if not variaveis_poluentes:
    variaveis_poluentes = [
        coluna for coluna in colunas_numericas if coluna.lower() not in VARIAVEIS_NAO_POLUENTES
    ]

if not variaveis_poluentes:
    raise RuntimeError("Nenhuma variavel de poluente foi identificada para o escopo.")

print("Variaveis de poluentes definidas para analise:")
print(variaveis_poluentes)


Variaveis de poluentes definidas para analise:
['pm10', 'pm2_5', 'o3', 'no2', 'so2', 'co', 'no', 'nox']


## 5. Criterios de qualidade minima (cobertura por variavel)

Calcula a cobertura de cada poluente no periodo definido e verifica se atende ao limite minimo.


In [5]:
# Considera um denominador por dia do calendario esperado para cobertura comparavel.
total_dias_referencia = len(calendario_esperado)

registros_cobertura = []
for variavel in variaveis_poluentes:
    dias_com_valor = int(df_rj[variavel].notna().sum())
    cobertura = dias_com_valor / total_dias_referencia if total_dias_referencia else 0.0

    registros_cobertura.append(
        {
            "variavel": variavel,
            "dias_com_valor": dias_com_valor,
            "dias_referencia": total_dias_referencia,
            "cobertura": cobertura,
            "cobertura_pct": cobertura * 100,
            "atende_criterio": cobertura >= COBERTURA_MINIMA,
        }
    )

df_cobertura = pd.DataFrame(registros_cobertura).sort_values(
    by=["atende_criterio", "cobertura"], ascending=[False, False]
)

variaveis_aprovadas = df_cobertura.loc[df_cobertura["atende_criterio"], "variavel"].tolist()
variaveis_reprovadas = df_cobertura.loc[~df_cobertura["atende_criterio"], "variavel"].tolist()

print(f"Criterio minimo de cobertura: {COBERTURA_MINIMA:.0%}")
print(f"Variaveis aprovadas: {len(variaveis_aprovadas)}")
print(f"Variaveis reprovadas: {len(variaveis_reprovadas)}")

display(df_cobertura)


Criterio minimo de cobertura: 80%
Variaveis aprovadas: 8
Variaveis reprovadas: 0


,variavel,dias_com_valor,dias_referencia,cobertura,cobertura_pct,atende_criterio
0,pm10,2534,2557,0.991005,99.100508,True
2,o3,2534,2557,0.991005,99.100508,True
3,no2,2534,2557,0.991005,99.100508,True
4,so2,2534,2557,0.991005,99.100508,True
5,co,2534,2557,0.991005,99.100508,True
6,no,2534,2557,0.991005,99.100508,True
7,nox,2534,2557,0.991005,99.100508,True
1,pm2_5,2516,2557,0.983966,98.396558,True


## 6. Auditoria de dados

Checagens de qualidade por variavel para apoiar a etapa de correlacao com internacoes.

Escopo da auditoria:
- Tipos de dados.
- Duplicatas por data no periodo.
- Valores ausentes.
- Valores fora de faixa valida (quando houver limite definido).
- Percentual de completude por variavel.
- Maior lacuna consecutiva de dados ausentes.


In [6]:
# Limites de referencia para detectar valores fora de faixa.
# Ajuste estes intervalos se houver padrao tecnico diferente no projeto.
FAIXAS_VALIDAS = {
    "pm10": (0, 600),
    "pm2_5": (0, 500),
    "o3": (0, 500),
    "no2": (0, 500),
    "so2": (0, 300),
    "co": (0, 50),
    "pts": (0, 1000),
    "no": (0, 500),
    "nox": (0, 500),
    "chuva": (0, 500),
    "tem": (-10, 50),
    "ur": (0, 100),
}

def maior_lacuna_consecutiva(mascara_ausente: pd.Series) -> int:
    # Calcula o maior bloco consecutivo de valores ausentes (True).
    if mascara_ausente.empty or not mascara_ausente.any():
        return 0

    grupos = (mascara_ausente != mascara_ausente.shift(fill_value=False)).cumsum()
    tamanhos = mascara_ausente.groupby(grupos).sum()
    return int(tamanhos.max())

# Reindexa por calendario diario para auditar completude/lacunas com base no periodo esperado.
base_calendario = (
    df_rj.set_index(COLUNA_DATA)
    .sort_index()
    .reindex(calendario_esperado)
)
base_calendario.index.name = COLUNA_DATA

registros_qualidade = []
total_dias_referencia = len(calendario_esperado)

for variavel in variaveis_poluentes:
    serie = base_calendario[variavel]

    # Metricas de tipo e completude.
    tipo_dado = str(df_rj[variavel].dtype)
    dias_com_dado = int(serie.notna().sum())
    dias_ausentes = int(total_dias_referencia - dias_com_dado)
    completude = (dias_com_dado / total_dias_referencia) if total_dias_referencia else 0.0

    # Metricas de lacuna temporal.
    lacuna_max_dias = maior_lacuna_consecutiva(serie.isna())

    # Metricas de valores fora de faixa (se houver limite definido).
    faixa = FAIXAS_VALIDAS.get(variavel)
    if faixa is not None:
        minimo, maximo = faixa
        mascara_fora_faixa = serie.notna() & ((serie < minimo) | (serie > maximo))
        qtd_fora_faixa = int(mascara_fora_faixa.sum())
        pct_fora_faixa = (qtd_fora_faixa / dias_com_dado) if dias_com_dado else 0.0
        faixa_referencia = f"[{minimo}, {maximo}]"
    else:
        qtd_fora_faixa = pd.NA
        pct_fora_faixa = pd.NA
        faixa_referencia = "nao_definida"

    registros_qualidade.append(
        {
            "variavel": variavel,
            "tipo_dado": tipo_dado,
            "dias_referencia": total_dias_referencia,
            "dias_com_dado": dias_com_dado,
            "dias_ausentes": dias_ausentes,
            "completude": completude,
            "completude_pct": completude * 100,
            "atende_completude": completude >= COBERTURA_MINIMA,
            "lacuna_max_consecutiva_dias": lacuna_max_dias,
            "faixa_referencia": faixa_referencia,
            "qtd_fora_faixa": qtd_fora_faixa,
            "pct_fora_faixa": pct_fora_faixa,
            "pct_fora_faixa_pct": (pct_fora_faixa * 100) if pd.notna(pct_fora_faixa) else pd.NA
        }
    )

# Entregavel principal da auditoria: tabela consolidada de qualidade dos dados.
tabela_qualidade_dados = pd.DataFrame(registros_qualidade).sort_values(
    by=["atende_completude", "completude", "lacuna_max_consecutiva_dias"],
    ascending=[False, False, True],
)
print(f"Criterio minimo de completude aplicado: {COBERTURA_MINIMA:.0%}")

display(tabela_qualidade_dados)


Criterio minimo de completude aplicado: 80%


,variavel,tipo_dado,dias_referencia,dias_com_dado,dias_ausentes,completude,completude_pct,atende_completude,lacuna_max_consecutiva_dias,faixa_referencia,qtd_fora_faixa,pct_fora_faixa,pct_fora_faixa_pct
0,pm10,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 600]",0,0.0,0.0
2,o3,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 500]",0,0.0,0.0
3,no2,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 500]",0,0.0,0.0
4,so2,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 300]",0,0.0,0.0
5,co,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 50]",0,0.0,0.0
6,no,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 500]",0,0.0,0.0
7,nox,float64,2557,2534,23,0.991005,99.100508,True,12,"[0, 500]",0,0.0,0.0
1,pm2_5,float64,2557,2516,41,0.983966,98.396558,True,17,"[0, 500]",0,0.0,0.0


## 7. Resumo do escopo definido

Consolida os parametros que serao usados nas proximas etapas da analise.



In [7]:
escopo_analise = {
    "periodo_inicio": df_rj[COLUNA_DATA].min(),
    "periodo_fim": df_rj[COLUNA_DATA].max(),
    "unidade_temporal": "diaria",
    "total_dias_esperados": len(calendario_esperado),
    "dias_observados": len(datas_observadas),
    "dias_faltantes": len(datas_faltantes),
    "criterio_cobertura_minima": COBERTURA_MINIMA,
    "variaveis_poluentes_identificadas": variaveis_poluentes,
    "variaveis_aprovadas": variaveis_aprovadas,
    "variaveis_reprovadas": variaveis_reprovadas,
}


print("Resumo do escopo definido:")
for chave, valor in escopo_analise.items():
    print(f"- {chave}: {valor}")


Resumo do escopo definido:
- periodo_inicio: 2012-01-01 00:00:00
- periodo_fim: 2018-12-31 00:00:00
- unidade_temporal: diaria
- total_dias_esperados: 2557
- dias_observados: 2557
- dias_faltantes: 0
- criterio_cobertura_minima: 0.8
- variaveis_poluentes_identificadas: ['pm10', 'pm2_5', 'o3', 'no2', 'so2', 'co', 'no', 'nox']
- variaveis_aprovadas: ['pm10', 'o3', 'no2', 'so2', 'co', 'no', 'nox', 'pm2_5']
- variaveis_reprovadas: []
